## upload images files


> **Running outside Google Colab?**  
> This notebook has been adapted for use in any Jupyter environment (local Jupyter, VS Code, JupyterHub, etc.).  
> Colab-specific cells (file upload, Google Drive mount, secret manager) have been replaced with portable equivalents.  
> See the comments in each affected cell for instructions.
>
> To use your own data, place files in a `data/` folder next to this notebook, or set the `DATA_FOLDER`
> environment variable to the path of your data directory.  
> Store API keys in a `.env` file (requires `pip install python-dotenv`) or set them as environment variables.


In [ ]:
# from google.colab import files  # Colab only — replaced below
# Provide local file paths instead:
from pathlib import Path
uploaded = {p.name: p.read_bytes() for p in Path('data/').glob('*') if p.is_file()}

In [ ]:
# needed to run async in Jupyter

import nest_asyncio
nest_asyncio.apply()

In [ ]:
%pip install -q pillow-heif

In [ ]:
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
from pillow_heif import register_heif_opener

# Register the HEIF opener
register_heif_opener()

drive_heic = Path("/content/").glob('**/*.HEIC')

for heic_file in tqdm(drive_heic):
  img = Image.open(heic_file)
  img.save(f"{heic_file.parent}/{heic_file.stem}.jpg")


In [ ]:
# from google.colab import userdata  # Colab only — replaced below
import os
# Load API keys from environment variables or a .env file:
# pip install python-dotenv  then add your key to a .env file
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # install python-dotenv if you use a .env file

# API Configuration
base_url = "https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
api_key = os.environ.get('DASHSCOPE_API_KEY')
model = "qwen3-vl-plus"  # or "qwen3-vl-flash" for faster/cheaper

In [ ]:
# show the image
from IPython.display import Image
Image(filename='/content/Pages from 1897-98 Spiski Sample Pages.jpg')

In [ ]:
# @title Image Encoding Function
import base64
from pathlib import Path

def encode_images(image_directory: str, file_extensions: list = ['.jpg', '.png'], data_uri: bool = True):
    """Lazy generator that encodes images on-demand.

    Args:
        image_directory: Path to directory containing images
        file_extensions: List of file extensions to process
        data_uri: Include data URI scheme in output

    Yields:
        tuple: (Path, base64_string) for each image
    """
    image_directory = Path(image_directory)
    if not image_directory.is_dir():
        raise ValueError(f"{image_directory} is not a valid directory")

    for path in image_directory.rglob('*'):
        if path.is_file() and path.suffix.lower() in file_extensions:
            with open(path, 'rb') as img_file:
                encoded = base64.b64encode(img_file.read()).decode('utf-8')
                if data_uri:
                    mime = 'image/jpeg' if path.suffix.lower() == '.jpg' else 'image/png'
                    yield path, f"data:{mime};base64,{encoded}"
                else:
                    yield path, encoded

In [ ]:
# @title OCR Processing Functions
import asyncio
import aiohttp
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm

client = AsyncOpenAI(api_key=api_key, base_url=base_url)

async def transcribe_image(img: tuple, session: aiohttp.ClientSession, semaphore: asyncio.Semaphore):
    """Process a single image with rate limiting."""
    async with semaphore:
        image_path, image_b64 = img

        try:
            completion = await asyncio.wait_for(
                client.chat.completions.create(
                    model=model,
                    messages=[{
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": "Extract text and return this table as structured data JSON blob."
                            },
                            {
                                "type": "image_url",
                                "image_url": {"url": image_b64}
                            }
                        ]
                    }],
                ),
                timeout=360
            )

            return {
                "name": image_path.name,
                "text": completion.choices[0].message.content
            }
        except asyncio.TimeoutError:
            print(f"⏱️  Timeout: {image_path.name}")
            return None
        except Exception as e:
            print(f"❌ Error on {image_path.name}: {e}")
            return None

async def transcribe_all_images(images, max_concurrent=15):
    """Process all images with concurrency control."""
    connector = aiohttp.TCPConnector(limit=10, limit_per_host=5)
    timeout = aiohttp.ClientTimeout(total=300)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        semaphore = asyncio.Semaphore(max_concurrent)

        # Encode images with progress
        print("📊 Encoding images...")
        images_list = list(tqdm(images, desc="Encoding"))

        # Process with progress
        tasks = [transcribe_image(img, session, semaphore) for img in images_list]
        results = []

        for task in tqdm.as_completed(tasks, total=len(tasks), desc="Transcribing"):
            result = await task
            if result:
                results.append(result)

    return results

In [ ]:
# @title Clean text helper functions
import re

def remove_code_tags(text):
    """Remove markdown code fences."""
    return re.sub(r"```[a-zA-Z]*\n(.*?)\n```", r"\1", text, flags=re.DOTALL)

def remove_repeated_phrases(text):
    """Remove consecutive repeated phrases."""
    pattern = r"(\b\w+(?: \w+){0,5}\b)( \1)+"
    return re.sub(pattern, r"\1", text)

def clean_text(text):
    """Clean extracted text."""
    text = remove_code_tags(text)
    text = remove_repeated_phrases(text)
    return text.strip()

In [ ]:
# Run the full pipeline
print("🚀 Starting OCR pipeline...")

# Step 1: Generate encoded images
images_gen = encode_images("/content/")

# Step 2: Process all images
results = asyncio.run(transcribe_all_images(images_gen, max_concurrent=15))

# Step 3: Clean the text
from tqdm import tqdm
for result in tqdm(results, desc="Cleaning text"):
    result["text"] = clean_text(result["text"])

print(f"✅ Processed {len(results)} images successfully")

In [ ]:
resilt

In [ ]:
#result.text to pd dataframe
import pandas as pd

df = pd.DataFrame(results)
df.head()

In [ ]:
# Save results
import json
import pandas as pd

# Save as JSON
output_json = "/content/" + "ocr_results.json"
with open(output_json, 'w') as f:
    json.dump(results, f, indent=2)

# Save as CSV
df = pd.DataFrame(results)
output_csv = "/content/" + "ocr_results.csv"
df.to_csv(output_csv, index=False)

print(f"💾 Saved results:")
print(f"   JSON: {output_json}")
print(f"   CSV: {output_csv}")

In [ ]:
for item in results:
  md_filename = "/content/" + item["name"].split(".")[0] + ".md"
  Path(md_filename).write_text(item["text"])

In [ ]:
from google.colab import files

txt_files = list(Path("/content/").glob("**/*.md"))
print(f"Found {len(txt_files)} markdown files")
for file in txt_files:
  files.download(file)


In [ ]:
data = {""}